In [6]:
# --- Prepare tensors and model (loads CSVs, builds per-sample PyG Dataset with train/test split)
import json
import torch
import config
import pandas as pd
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from c21_surrogate_model_v3 import create_model, FocalLoss

edge_csv = "v5_edge_C12_S9999_D20260430"
node_csv = "v5_node_C12_S9999_D20260430"

node_csv_path = config.GH_DATA_PATH / f"{node_csv}.csv"
edge_csv_path = config.GH_DATA_PATH / f"{edge_csv}.csv"

nodes_df = pd.read_csv(node_csv_path)
edges_df = pd.read_csv(edge_csv_path)

# Load topology (edge_index) saved as a JSON list [[src...],[dst...]]
edge_index_path = Path(config.DATA_IO_PATH) / "edge_index.json"
if not edge_index_path.exists():
    raise FileNotFoundError(
        f"edge_index.json not found at {edge_index_path}. Provide a valid topology file."
    )
with open(edge_index_path, "r") as f:
    edge_index_list = json.load(f)
edge_index = torch.tensor(edge_index_list, dtype=torch.long)

if edge_index.ndim != 2 or edge_index.shape[0] != 2:
    raise ValueError(
        f"edge_index must have shape [2, num_edges], got {tuple(edge_index.shape)}"
    )

# REQUIRED columns (no synthetic/fallback values allowed)
node_cols = ["x", "y", "z", "Tx", "Ty", "Tz", "Rx", "Ry", "Rz", "Fz"]
edge_cols = ["Area", "Length", "E", "Iy", "Iz", "J", "EA/L"]

missing_node_cols = [c for c in node_cols if c not in nodes_df.columns]
if missing_node_cols:
    raise KeyError(
        f"Missing required node columns: {missing_node_cols}. "
        f"No synthetic values will be created; please provide these columns in {node_csv_path}."
    )

missing_edge_cols = [c for c in edge_cols if c not in edges_df.columns]
if missing_edge_cols:
    raise KeyError(
        f"Missing required edge columns: {missing_edge_cols}. "
        f"No synthetic values will be created; please provide these columns in {edge_csv_path}."
    )

if "Utilization" not in edges_df.columns:
    raise KeyError(
        f"Missing required target column 'Utilization' in {edge_csv_path}. "
        "Provide Utilization for supervised training."
    )

# Per-sample dataset construction
num_edges = int(edge_index.shape[1])
expected_num_nodes = int(edge_index.max().item()) + 1
print(f"Topology check: edge_index has {num_edges} edges, max node id {expected_num_nodes-1} => expecting {expected_num_nodes} nodes.")

dataset = []

for col in ("sample_id", "Sample_ID", "SampleId"):
    if col in nodes_df.columns and col in edges_df.columns:
        sample_id_col = col
        break

if sample_id_col:
    # Group and validate counts per sample
    node_groups = nodes_df.groupby(sample_id_col)
    edge_groups = edges_df.groupby(sample_id_col)
    samples = sorted(set(node_groups.groups.keys()).intersection(edge_groups.groups.keys()))
    if not samples:
        raise ValueError("No matching sample IDs between node and edge CSVs.")
    for s in samples:
        n_df = node_groups.get_group(s)
        e_df = edge_groups.get_group(s)
        if len(n_df) != expected_num_nodes:
            raise ValueError(f"Sample {s}: node count {len(n_df)} != expected {expected_num_nodes}")
        if len(e_df) != num_edges:
            raise ValueError(f"Sample {s}: edge count {len(e_df)} != expected {num_edges}")
        x = torch.tensor(n_df[node_cols].values, dtype=torch.float32)
        edge_attr = torch.tensor(e_df[edge_cols].values, dtype=torch.float32)
        y = torch.tensor((e_df['Utilization'] > 1).astype(int).values, dtype=torch.float32).view(-1,1)
        dataset.append(Data(x=x, edge_index=edge_index.clone(), edge_attr=edge_attr, y=y))

if not dataset:
    raise RuntimeError("No samples found when constructing dataset — check CSV formats and edge_index.")

print(f"Constructed dataset with {len(dataset)} samples; each has {expected_num_nodes} nodes and {num_edges} edges.")

# Train/Test Split: 80% train, 10% val, 10% test
torch.manual_seed(42)  # For reproducibility
indices = torch.randperm(len(dataset)).tolist()
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

train_dataset = [dataset[i] for i in train_indices]
val_dataset = [dataset[i] for i in val_indices]
test_dataset = [dataset[i] for i in test_indices]

print(f"Split: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}")

# Instantiate model (do not cache topology for batched training)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = create_model(node_features_dim=len(node_cols), edge_features_dim=len(edge_cols), device=device)

# DataLoaders
batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoaders ready:")
print(f"  Train: {len(train_dataloader)} batches (batch_size={batch_size})")
print(f"  Val:   {len(val_dataloader)} batches")
print(f"  Test:  {len(test_dataloader)} batches")

# Move model to device
model.to(device)


Topology check: edge_index has 120 edges, max node id 38 => expecting 39 nodes.
Constructed dataset with 10000 samples; each has 39 nodes and 120 edges.
Split: Train=8000 | Val=1000 | Test=1000
DataLoaders ready:
  Train: 250 batches (batch_size=32)
  Val:   32 batches
  Test:  32 batches


TrussEdgeSafetyGNN(
  (node_encoder): Linear(in_features=10, out_features=128, bias=True)
  (nnconv_layers): ModuleList(
    (0-3): 4 x NNConv(128, 128, aggr=add, nn=EdgeFeatureMLPFilter(
      (fc1): Linear(in_features=7, out_features=64, bias=True)
      (fc2): Linear(in_features=64, out_features=16384, bias=True)
      (activation): LeakyReLU(negative_slope=0.1)
    ))
  )
  (batch_norms): ModuleList(
    (0-3): 4 x BatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (activation): LeakyReLU(negative_slope=0.1)
  (edge_decoder): EdgeDecoder(
    (fc1): Linear(in_features=263, out_features=128, bias=True)
    (fc2): Linear(in_features=128, out_features=64, bias=True)
    (fc3): Linear(in_features=64, out_features=1, bias=True)
    (activation): LeakyReLU(negative_slope=0.1)
    (sigmoid): Sigmoid()
  )
)

In [ ]:
# --- Training loop using train_dataloader (per-sample batches) ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = FocalLoss(alpha=0.1, gamma=2.0)

epochs = 10
train_losses = []
val_losses = []

model.train()
for epoch in range(epochs):
    # Training phase
    epoch_train_loss = 0.0
    for batch in train_dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()
        preds = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = loss_fn(preds, batch.y)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch.num_graphs
    epoch_train_loss /= len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation phase (optional monitoring)
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for batch in val_dataloader:
            batch = batch.to(device)
            preds = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = loss_fn(preds, batch.y)
            epoch_val_loss += loss.item() * batch.num_graphs
    epoch_val_loss /= len(val_dataset) if len(val_dataset) > 0 else 1
    val_losses.append(epoch_val_loss)
    model.train()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:03d}  train_loss={epoch_train_loss:.6f}  val_loss={epoch_val_loss:.6f}')

# Save model checkpoint
torch.save({'model_state_dict': model.state_dict()}, 'surrogate_v3_checkpoint.pth')
print('Checkpoint saved: surrogate_v3_checkpoint.pth')

# Evaluate on test set (unseen during training)
print("\nEvaluating on test set...")
model.eval()
test_loss = 0.0
all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        preds = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = loss_fn(preds, batch.y)
        test_loss += loss.item() * batch.num_graphs
        all_test_preds.append(preds.cpu())
        all_test_targets.append(batch.y.cpu())

test_loss /= len(test_dataset) if len(test_dataset) > 0 else 1
print(f'Test set loss: {test_loss:.6f}')

# Concatenate predictions and targets
test_preds = torch.cat(all_test_preds, dim=0).numpy()
test_targets = torch.cat(all_test_targets, dim=0).numpy()
print(f'Test predictions shape: {test_preds.shape}, Test targets shape: {test_targets.shape}')


### Visual

In [ ]:
# Plot: Loss curves + test predictions for binary classification
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score, accuracy_score

# Create epoch history
epochs_range = np.arange(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Training and validation loss curves
ax = axes[0]
ax.plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2, marker='o', markersize=4)
ax.plot(epochs_range, val_losses, 'orange', label='Val Loss', linewidth=2, marker='s', markersize=4)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss (Focal Loss)', fontsize=11)
ax.set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Prediction probability distribution
ax = axes[1]
test_preds_flat = test_preds.flatten()
test_targets_flat = test_targets.flatten()
ax.hist(test_preds_flat[test_targets_flat == 0], bins=25, alpha=0.6, label='Safe (target=0)', color='green', edgecolor='black')
ax.hist(test_preds_flat[test_targets_flat == 1], bins=25, alpha=0.6, label='Unsafe (target=1)', color='red', edgecolor='black')
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Prediction Distribution by Class', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: ROC Curve
ax = axes[2]
fpr, tpr, _ = roc_curve(test_targets_flat, test_preds_flat)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', linewidth=2, linestyle='--', label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary metrics
test_preds_binary = (test_preds_flat > 0.5).astype(int)
accuracy = accuracy_score(test_targets_flat, test_preds_binary)
tn, fp, fn, tp = confusion_matrix(test_targets_flat, test_preds_binary).ravel()

print(f"\n✅ Training complete!")
print(f"Final Train Loss: {train_losses[-1]:.6f}")
print(f"Final Val Loss:   {val_losses[-1]:.6f}")
print(f"\nTest Set Metrics (threshold=0.5):")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  ROC AUC:   {roc_auc:.4f}")
print(f"  True Positives:  {tp} | True Negatives:  {tn}")
print(f"  False Positives: {fp} | False Negatives: {fn}")


# EVALUATION

In [ ]:
# Predictions vs Actual + Residual diagnostics (Train/Test)
import matplotlib.pyplot as plt
import numpy as np
import torch
import config
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Environment ready for visualization.")

from src.c21_diagnostic_verify_r2 import collect_preds_trues_original, drop_non_finite_pairs, compute_split_metrics
from c21_model_evaluation_v3 import build_pred_residual_figure, build_error_distribution_figure

# Collect predictions in original kN scale
train_preds_original, train_trues_original = collect_preds_trues_original(
    train_loader,
    model,
    device,
    edge_target_scaler,
)
test_preds_original, test_trues_original = collect_preds_trues_original(
    test_loader,
    model,
    device,
    edge_target_scaler,
)

# Remove NaN/Inf values pairwise before residuals/metrics
train_trues_original, train_preds_original = drop_non_finite_pairs(
    train_trues_original,
    train_preds_original,
    "train",
)
test_trues_original, test_preds_original = drop_non_finite_pairs(
    test_trues_original,
    test_preds_original,
    "test",
)

# Residuals and metrics
train_residuals = train_trues_original - train_preds_original
test_residuals = test_trues_original - test_preds_original

train_metrics = compute_split_metrics(train_trues_original, train_preds_original)
test_metrics = compute_split_metrics(test_trues_original, test_preds_original)

train_r2 = train_metrics["r2"]
test_r2 = test_metrics["r2"]
train_mae = train_metrics["mae"]
test_mae = test_metrics["mae"]
train_rmse = train_metrics["rmse"]
test_rmse = test_metrics["rmse"]

print(f"Train R2:  {train_r2:.4f}")
print(f"Test R2:   {test_r2:.4f}")
print(f"Train MAE: {train_mae:.4f} kN")
print(f"Test MAE:  {test_mae:.4f} kN")
print(f"Train RMSE: {train_rmse:.4f} kN")
print(f"Test RMSE:  {test_rmse:.4f} kN\n")

pred_residuals_fig = build_pred_residual_figure(
    train_trues_original,
    train_preds_original,
    test_trues_original,
    test_preds_original,
    train_r2,
    test_r2,
)

# Error distribution plots
error_dist_fig = build_error_distribution_figure(
    train_residuals,
    test_residuals,
    train_mae,
    test_mae,
 )


# EXPORT REPORT

In [ ]:
# Final evaluation export
import importlib
from c21_model_evaluation_v3 import save_evaluation
from c00_naming import build_model_artifact_stem

# Determine fit status
r2_gap = train_r2 - test_r2
if train_r2 < 0.7 and test_r2 < 0.7:
    status = "underfitting"
elif r2_gap > 0.05:
    status = "overfitting"
else:
    status = "good_fit"

metrics = {
    "train_r2": train_r2,
    "test_r2": test_r2,
    "train_mae": train_mae,
    "test_mae": test_mae,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "r2_gap": float(r2_gap),
}

# Optional training figure from the first visualization cell
training_visuals_fig_local = training_visuals_fig if "training_visuals_fig" in globals() else None

# Keep ID consistent with model/scaler exports in 01_surrogate_models
model_artifact_stem = build_model_artifact_stem(
    params["run_id"],
    params["learning_rate"],
    params["epochs"],
    final_val_r2,
)

architecture_summary = {
    "model_class": "TrussEdgeNNConv",
    "node_in_dim": len(schema.node_continuous_cols) + len(schema.node_mask_cols),
    "edge_in_dim": len(schema.edge_feature_cols),
    "global_in_dim": len(schema.global_feature_cols),
    "selected_node_continuous_cols": tuple(schema.node_continuous_cols),
    "selected_node_mask_cols": tuple(schema.node_mask_cols),
    "selected_node_virtual_cols": tuple(getattr(schema, "node_virtual_cols", ())),
    "selected_edge_feature_cols": tuple(schema.edge_feature_cols),
    "selected_global_feature_cols": tuple(schema.global_feature_cols),
    "node_features": tuple(schema.node_continuous_cols) + tuple(schema.node_mask_cols) + tuple(getattr(schema, "node_virtual_cols", ())),
    "edge_features": tuple(schema.edge_feature_cols),
    "global_features": tuple(schema.global_feature_cols),
    "hidden_dim": params["hidden_dim"],
    "edge_count": schema.edge_count,
    "device": str(device),
    "dataset_sources": {
        "node": params["node_csv"],
        "edge": params["edge_csv"],
        "global": params["global_csv"],
    },
}

experiment_notes = (
    f"USE_PRETRAINED={params['use_pretrained']}; "
    f"lr={params['learning_rate']}; epochs={params['epochs']}; "
    f"batch_size={params['batch_size']}; hidden_dim={params['hidden_dim']}; "
    f"weight_decay={params['weight_decay']}"
)

# Get epoch metrics history from training results
epoch_metrics_history = results.get("epoch_metrics_history", []) if "results" in globals() else []

# Save evaluation
saved_files = save_evaluation(
    model_prefix=model_artifact_stem,
    dataset_name=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    metrics=metrics,
    pred_residuals_fig=pred_residuals_fig,
    error_dist_fig=error_dist_fig,
    training_visuals_fig=training_visuals_fig_local,
    node_count=schema.node_count,
    edge_count=schema.edge_count,
    export_path=config.SM_DATA_PATH,
    status=status,
    run_id=params["run_id"],
    artifact_stem=model_artifact_stem,
    feature_count=architecture_summary["node_in_dim"] + architecture_summary["edge_in_dim"] + architecture_summary["global_in_dim"],
    learning_rate=params["learning_rate"],
    epochs=params["epochs"],
    eval_every=params.get("eval_every"),
    final_val_r2=final_val_r2,
    strict_dataset_label=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    source_dataset_path=str(config.GH_DATA_PATH / params["edge_csv"]),
    architecture_summary=architecture_summary,
    experiment_notes=experiment_notes,
    train_split_ratio=params["train_split_ratio"],
    random_seed=params["random_seed"],
    source_notebook="c21_surrogate_model_training.ipynb",
    epoch_metrics_history=epoch_metrics_history,
)

print(f"\n✅ Evaluation exported:")
for f in saved_files:
    print(f"  - {f}")